<a href="https://colab.research.google.com/github/aabyyaann/midterm-machine-learning/blob/main/UTS_ML_NaufalAlifAbyan_101032300032.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# UTS Machine Learning - Naufal Alif Abyan (101032300032)
!pip install mlflow optuna lightgbm -q

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import optuna
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

print("Library siap digunakan.")

Library siap digunakan.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

def find_data_path(file_name):
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if file_name in files:
            return os.path.join(root, '')
    return None

# Mencari folder dataset secara otomatis
PATH = find_data_path('train_transaction.csv')

if PATH:
    print(f"Lokasi data: {PATH}")
    # Memuat data dengan low_memory=False untuk stabilitas koneksi
    train_trans = pd.read_csv(PATH + 'train_transaction.csv', low_memory=False)

    try:
        train_id = pd.read_csv(PATH + 'train_identity.csv', low_memory=False)
        # Menggabungkan tabel sesuai instruksi soal
        df = pd.merge(train_trans, train_id, on='TransactionID', how='left')
        print("Data Transaction dan Identity berhasil digabung.")
    except:
        df = train_trans
        print("Hanya menggunakan data Transaction.")

    # Optimasi memori
    for col in df.columns:
        if df[col].dtype == 'float64': df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64': df[col] = df[col].astype('int32')
else:
    print("File tidak ditemukan. Pastikan file sudah berada di Google Drive.")

Mounted at /content/drive
Lokasi data: /content/drive/MyDrive/Midterm ML/


In [ ]:
# Membatasi jumlah fitur untuk efisiensi komputasi
X = df.iloc[:, :50].drop(['isFraud', 'TransactionID'], axis=1, errors='ignore')
y = df['isFraud']

# Preprocessing: Mengisi nilai kosong dan Encoding
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].fillna('Unknown')
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    else:
        X[col] = X[col].fillna(X[col].median())

# Split data dengan stratifikasi (menjaga rasio label fraud)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'is_unbalance': True, # Penanganan class imbalance sesuai soal
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 31, 100),
        'verbosity': -1
    }

    with mlflow.start_run(nested=True):
        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train)
        preds = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, preds)

        mlflow.log_params(params)
        mlflow.log_metric("auc", auc)
        return auc

# Menjalankan optimasi
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)

print(f"Skor terbaik: {study.best_value}")

In [ ]:
# Melatih model final dengan parameter terbaik
best_model = lgb.LGBMClassifier(**study.best_params)
best_model.fit(X_train, y_train)

# Hasil prediksi
y_pred_prob = best_model.predict_proba(X_val)[:, 1]
y_pred_bin = (y_pred_prob > 0.5).astype(int)

# Tampilan evaluasi
print(f"\nFinal AUC Score: {roc_auc_score(y_val, y_pred_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_bin))

# Grafik Confusion Matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_val, y_pred_bin), annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {NIM}')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.show()